# 25 — Advanced Time-Series Indexing, Frequency Grids, & Resampling Windows
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive guide to high-frequency date ranges, partial string date slicing, resampling interval boundaries (`closed` vs `label`), and Period vs Timestamp architecture in Pandas.*

---

## 📌 Executive Summary & Interview Expectations
In quantitative finance, IoT monitoring, and distributed systems telemetry, handling continuous time-series data requires precision. In technical interviews, examiners focus on:
1. **Partial String Date Indexing**: How Pandas dynamically resolves string queries like `df.loc['2023-05']` or range slices on DatetimeIndex.
2. **Frequency Generation (`pd.date_range`)**: Building exact temporal grids across business days (`'B'`), month ends (`'ME'`), and high-frequency minutes (`'min'`).
3. **Resampling Mechanics (`closed` vs `label`)**: The exact interval math governing time buckets and preventing forward-looking data leakage.
4. **`pd.Timestamp` (Points in Time) vs `pd.Period` (Spans of Time)**: Memory representations, arithmetic differences, and conversion methods.
5. **Modern API Standards**: Eliminating deprecated aliases (replacing `'T'` with `'min'`, replacing `'M'` with `'ME'`, replacing `errors='ignore'`).

## 1. Python `datetime` vs Pandas `Timestamp`

In [1]:
import datetime as dt
import numpy as np
import pandas as pd

# Python standard datetime
today = dt.date(2022, 12, 22)
now = dt.datetime(2022, 12, 22, 17, 35, 20)

print(f"dt.date: {today} | Year: {today.year}, Month: {today.month}, Day: {today.day}")
print(f"dt.datetime: {now} | Hour: {now.hour}, Minute: {now.minute}, Second: {now.second}")

# Pandas Timestamp
ts = pd.Timestamp("2022-12-22 17:35:20")
print(f"pd.Timestamp: {ts} (dtype: {type(ts)})")

dt.date: 2022-12-22 | Year: 2022, Month: 12, Day: 22
dt.datetime: 2022-12-22 17:35:20 | Hour: 17, Minute: 35, Second: 20
pd.Timestamp: 2022-12-22 17:35:20 (dtype: <class 'pandas.Timestamp'>)


## 2. Generating DatetimeIndex & Robust Parsing

### 💡 Interview Tip: Deprecation of `errors='ignore'`
In modern Pandas 2.2+ / 3.0, `errors='ignore'` is deprecated in `pd.to_datetime`.
- If parsing succeeds: returns `DatetimeIndex`.
- If parsing encounters malformed data: use `errors='coerce'` to produce `NaT` (Not-a-Time), or `errors='raise'` (default) to catch corrupt data early.

In [2]:
raw_dates = ["2022-12-22", "2022-12-23", "2022-12-24", "corrupt_date"]

# Coerce invalid dates into NaT
clean_dt_index = pd.to_datetime(raw_dates, errors="coerce")
print("Parsed DatetimeIndex:")
print(clean_dt_index)

Parsed DatetimeIndex:
DatetimeIndex(['2022-12-22', '2022-12-23', '2022-12-24', 'NaT'], dtype='datetime64[us]', freq=None)


## 3. Temporal Grid Generation: `pd.date_range()`

### Key Frequency Parameters:
- `'D'`: Calendar day
- `'B'`: Business day (skips weekends)
- `'W-MON'`: Weekly on Mondays
- `'h'`: Hourly (replaces legacy `'H'`)
- `'min'`: Minute intervals (replaces legacy `'T'` / `'MIN'` / `'S'` frequency warnings)

In [3]:
# Daily range
daily_grid = pd.date_range(start="2024-01-01", periods=5, freq="D")
print("Calendar Days:", daily_grid.strftime("%Y-%m-%d").tolist())

# Business Day range (skips Saturday and Sunday)
bday_grid = pd.date_range(start="2024-01-05", periods=5, freq="B")
print("Business Days (Notice skip across weekend):", bday_grid.strftime("%Y-%m-%d (%a)").tolist())

# High-frequency minute grid
minute_grid = pd.date_range(start="2024-01-01 09:30:00", periods=4, freq="15min")
print("15-Minute Grid:", minute_grid.strftime("%H:%M:%S").tolist())

Calendar Days: ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05']
Business Days (Notice skip across weekend): ['2024-01-05 (Fri)', '2024-01-08 (Mon)', '2024-01-09 (Tue)', '2024-01-10 (Wed)', '2024-01-11 (Thu)']
15-Minute Grid: ['09:30:00', '09:45:00', '10:00:00', '10:15:00']


## 4. Time-Series Slicing & Partial String Indexing

### 💡 Interview Tip: Human-Readable Partial String Slicing
When a DataFrame is indexed by a `DatetimeIndex`:
- `df.loc['2023']`: Extracts the entire year 2023.
- `df.loc['2023-05']`: Extracts the entire month of May 2023.
- `df.loc['2023-05-01':'2023-05-15']`: Slices the exact date range (both endpoints included).
- `.truncate(before=..., after=...)`: Rapid bounding without manual slicing syntax.

In [4]:
# Construct synthetic stock time series
np.random.seed(42)
dates_2023 = pd.date_range("2023-01-01", "2023-12-31", freq="B")
prices = 150 + np.cumsum(np.random.normal(0, 1.5, size=len(dates_2023)))

stock_ts = pd.DataFrame({"Close": prices.round(2)}, index=dates_2023)
print(f"Generated Stock Series with {len(stock_ts)} trading days.")

# Partial string slice for December 2023
december_data = stock_ts.loc["2023-12"]
print(f"\nTrading days in December 2023: {len(december_data)}")
display(december_data.head(3))

# Truncate to a specific window
truncated = stock_ts.truncate(before="2023-12-05", after="2023-12-15")
print(f"\nTruncated Window (Dec 5 to Dec 15): {len(truncated)} trading days")
display(truncated)

Generated Stock Series with 260 trading days.

Trading days in December 2023: 21


,Close
2023-12-01,149.14
2023-12-04,147.95
2023-12-05,147.77



Truncated Window (Dec 5 to Dec 15): 9 trading days


,Close
2023-12-05,147.77
2023-12-06,148.53
2023-12-07,149.83
2023-12-08,148.03
2023-12-11,147.53
2023-12-12,146.82
2023-12-13,145.84
2023-12-14,148.48
2023-12-15,149.09


## 5. String Parsing vs String Formatting: `strptime` vs `strftime`

### 💡 Memory Hook:
- **`strptime` (Parse)**: String $\rightarrow$ Datetime object (*"p for parse"*).
- **`strftime` (Format)**: Datetime object $\rightarrow$ String (*"f for format"*).

In [5]:
# 1. strptime: Parse raw text into structured datetime
raw_date_str = "2024/07/23 04:32:51"
parsed_dt = dt.datetime.strptime(raw_date_str, "%Y/%m/%d %H:%M:%S")
print(f"strptime result: {parsed_dt} (Type: {type(parsed_dt)})")

# 2. strftime: Format datetime into customized display string
formatted_str = parsed_dt.strftime("%A, %B %d, %Y at %I:%M %p")
print(f"strftime result: '{formatted_str}' (Type: {type(formatted_str)})")

strptime result: 2024-07-23 04:32:51 (Type: <class 'datetime.datetime'>)
strftime result: 'Tuesday, July 23, 2024 at 04:32 AM' (Type: <class 'str'>)


## 6. Resampling Boundaries: Demystifying `closed` and `label`

### 🚨 Top Interview Gotcha: The Resampling Boundary Matrix
When downsampling high-frequency data into 5-minute bins:
- `closed='left'`: The bin is $[09:00, 09:05)$ (includes 09:00, excludes 09:05).
- `closed='right'`: The bin is $(09:00, 09:05]$ (excludes 09:00, includes 09:05).
- `label='left'`: The bin is named after its start time (`09:00`).
- `label='right'`: The bin is named after its end time (`09:05`).

*Financial markets standard*: Candlestick charts typically use `closed='left'` and `label='left'`!

In [6]:
# Create 1-minute synthetic telemetry data
min_index = pd.date_range("2024-02-01 09:00:00", periods=10, freq="min")
telemetry = pd.DataFrame({"Metric": range(10)}, index=min_index)
print("Raw 1-Minute Telemetry:")
display(telemetry)

print("\n5-Minute Downsample (closed='left', label='left') [Standard]:")
display(telemetry.resample("5min", closed="left", label="left").sum())

print("\n5-Minute Downsample (closed='right', label='right'):")
display(telemetry.resample("5min", closed="right", label="right").sum())

Raw 1-Minute Telemetry:


,Metric
2024-02-01 09:00:00,0
2024-02-01 09:01:00,1
2024-02-01 09:02:00,2
2024-02-01 09:03:00,3
2024-02-01 09:04:00,4
2024-02-01 09:05:00,5
2024-02-01 09:06:00,6
2024-02-01 09:07:00,7
2024-02-01 09:08:00,8
2024-02-01 09:09:00,9



5-Minute Downsample (closed='left', label='left') [Standard]:


,Metric
2024-02-01 09:00:00,10
2024-02-01 09:05:00,35



5-Minute Downsample (closed='right', label='right'):


,Metric
2024-02-01 09:00:00,0
2024-02-01 09:05:00,15
2024-02-01 09:10:00,30


## 7. `pd.Timestamp` vs `pd.Period`

### Comparison Matrix
| Feature | `pd.Timestamp` | `pd.Period` |
| :--- | :--- | :--- |
| **Concept** | A specific instantaneous point in time | A fixed duration/span of time |
| **Example** | `2024-03-15 14:30:00` | `2024Q1` or `2024-03` |
| **Index Type** | `pd.DatetimeIndex` | `pd.PeriodIndex` |
| **Frequency Conversion**| Resampling | `.asfreq('M')`, `.to_timestamp()` |

In [7]:
# Create quarterly periods
quarter_periods = pd.period_range(start="2024-01-01", periods=4, freq="Q")
period_df = pd.DataFrame({"Revenue": [100, 120, 110, 150]}, index=quarter_periods)

print("Quarterly Period DataFrame:")
display(period_df)

# Convert PeriodIndex to DatetimeIndex
converted_ts = period_df.to_timestamp(how="start")
print("\nConverted to DatetimeIndex (start of quarter):")
display(converted_ts)

Quarterly Period DataFrame:


,Revenue
2024Q1,100
2024Q2,120
2024Q3,110
2024Q4,150



Converted to DatetimeIndex (start of quarter):


,Revenue
2024-01-01,100
2024-04-01,120
2024-07-01,110
2024-10-01,150


---
## 🎯 8. Technical Interview Corner: Tricky Questions & Drills

### Q1: What is the risk of using `label='right'` in production real-time trading pipelines?
**Answer**:
If data is arriving in real-time and an algorithm labels the bucket from 09:00 to 09:05 as `09:05`, but an analyst backtesting the strategy writes logic that acts at `09:00` on the `09:05` labeled row, it introduces **lookahead bias (data leakage)**! Always be crystal clear whether timestamps represent interval starts or interval ends.

---

### Q2: Advanced Interview Coding Challenge: Resampling Tick Data to Candlestick OHLCV
**Challenge**:
Given high-frequency irregular price ticks, generate **5-minute financial candlestick bars** computing:
- `Open`: first price in the 5-min window
- `High`: max price in the window
- `Low`: min price in the window
- `Close`: last price in the window
- `Volume`: sum of volume in the window

In [8]:
# Interview Solution: High-Performance OHLCV Bar Construction
# 1. Generate irregular tick arrivals
np.random.seed(101)
tick_times = pd.date_range("2024-01-01 09:30:00", "2024-01-01 10:00:00", freq="23s")
tick_prices = 100 + np.cumsum(np.random.normal(0, 0.2, len(tick_times)))
tick_volumes = np.random.randint(10, 200, size=len(tick_times))

tick_df = pd.DataFrame({"price": tick_prices.round(2), "volume": tick_volumes}, index=tick_times)

# 2. Resample into 5-minute OHLCV bars
ohlcv = tick_df.resample("5min", closed="left", label="left").agg({
    "price": ["first", "max", "min", "last"],
    "volume": "sum"
})
ohlcv.columns = ["Open", "High", "Low", "Close", "Volume"]

print("Constructed 5-Minute OHLCV Candlestick Bars:")
display(ohlcv)

Constructed 5-Minute OHLCV Candlestick Bars:


,Open,High,Low,Close,Volume
2024-01-01 09:30:00,100.54,101.08,100.54,100.59,1447
2024-01-01 09:35:00,100.40,102.08,100.40,101.59,1194
2024-01-01 09:40:00,101.63,102.46,101.56,102.46,1424
2024-01-01 09:45:00,102.26,102.80,102.07,102.49,1202
2024-01-01 09:50:00,102.39,103.60,102.39,103.60,1524
2024-01-01 09:55:00,103.60,103.98,103.25,103.54,1244
